# 🫁 Phase 1.1v4: Enhanced Image Classifier — Single Split + TTA (PA Viewpoint)

### Upgrades over v3:
1. **5-Fold Patient-Level Cross-Validation** (`GroupKFold` on `subject_id`) — robust metrics across all data
2. **Test Time Augmentation (TTA)** — average predictions over 10 augmented versions of each test image

### Retained from v3:
- DenseNet121 CXR-pretrained backbone (`torchxrayvision`)
- CLAHE preprocessing + Lung ROI cropping
- CBAM attention
- Focal Loss (γ=2) + 2× pneumonia class weight
- Mixup augmentation (α=0.3) + Label smoothing (0.1)
- Safe augmentations only (no horizontal flip)
- Differential learning rates
- Youden's J threshold tuning per fold

In [1]:
# --- CELL 0: Imports & Reproducibility ---
import os, random, warnings, copy, json as json_lib
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score, f1_score, auc,
    precision_recall_curve, average_precision_score
)

import torchxrayvision as xrv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', torch.cuda.get_device_properties(DEVICE).total_memory / 1e9, 'GB')

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Memory: 4.294508544 GB


In [ ]:
# --- CELL 1: Paths & Hyperparameters ---
BASE_DIR     = r'C:\2026\PneumoFusionNet\mimic\main'
DATASET_DIR  = os.path.join(BASE_DIR, 'dataset')
CSV_PATH     = os.path.join(DATASET_DIR, 'mimic_paired_dataset.csv')
SAVE_DIR     = os.path.join(BASE_DIR, 'outputs', 'Phase_1.1v4_PA_single_split_exp1')
BBOX_CSV     = os.path.join(BASE_DIR, 'outputs', 'Phase_1.1v4_PA_crossval_scaleup', 'lung_bboxes.csv')
SPLIT_CSV    = os.path.join(DATASET_DIR, 'patient_split.csv')

IMG_SIZE       = 224
BATCH_SIZE     = 8
NUM_EPOCHS     = 20
LR             = 3e-5
WEIGHT_DECAY   = 1e-4
NUM_CLASSES    = 2
WARMUP_EPOCHS  = 5
GRAD_CLIP      = 1.0
PATIENCE       = 10
MIXUP_ALPHA    = 0.3
LABEL_SMOOTH   = 0.1
PNEUMONIA_WT   = 2.0
TTA_TRANSFORMS = 10  # Number of augmented views for TTA

CLASSES = ['NORMAL', 'PNEUMONIA']
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save dir    : {SAVE_DIR}')
print(f'Dataset CSV : {CSV_PATH} (exists: {os.path.exists(CSV_PATH)})')
print(f'BBox CSV    : {BBOX_CSV} (exists: {os.path.exists(BBOX_CSV)})')
print(f'Split CSV   : {SPLIT_CSV} (exists: {os.path.exists(SPLIT_CSV)})')
print(f'Epochs      : {NUM_EPOCHS} | Batch size: {BATCH_SIZE} | LR: {LR}')

In [ ]:
# --- CELL 2: Data Loading & Filtering ---
df = pd.read_csv(CSV_PATH)

def resolve_data_path(p):
    if pd.isna(p) or str(p).strip() == '':
        return ''
    p = str(p).replace('/', os.sep).replace('\\', os.sep)
    return p if os.path.isabs(p) else os.path.join(DATASET_DIR, p)

df['image_path'] = df['image_path'].apply(resolve_data_path)
df['label'] = np.select(
    [df['pneumonia_label'].eq(1.0), df['no_finding_label'].eq(1.0)],
    [1, 0], default=-1
)
df = df[(df['label'] >= 0) & df['view_position'].eq('PA')].copy().reset_index(drop=True)
df['label_name'] = df['label'].map({0: 'Normal', 1: 'Pneumonia'})

missing = df['image_path'].apply(lambda p: not os.path.exists(p)).sum()
print(f'Total cohort   : {len(df):,} images ({df["subject_id"].nunique():,} unique patients)')
print(f'Label counts   : Normal={(df["label"]==0).sum():,}, Pneumonia={(df["label"]==1).sum():,}')
print(f'Missing images : {missing}')

In [ ]:
# --- CELL: Load Canonical Patient-Level Split (Experiment 1) ---
# All phases share the exact same patient split from patient_split.csv
import sys as _sys
PROJECT_ROOT = r'C:\2026\PneumoFusionNet'
if PROJECT_ROOT not in _sys.path:
    _sys.path.insert(0, PROJECT_ROOT)

from src.data.load_split import load_patient_split

train_df, val_df, test_df = load_patient_split(df, SPLIT_CSV)

print('=' * 70)
print('EXPERIMENT 1 — SHARED PATIENT-LEVEL SPLIT LOADED')
print('=' * 70)
for name, sdf in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n0 = (sdf['label']==0).sum(); n1 = (sdf['label']==1).sum()
    print(f'{name:5s}: {len(sdf):5d} rows | {sdf["subject_id"].nunique():4d} patients '
          f'(Normal={n0}, Pneumonia={n1}, {n1/len(sdf)*100:.1f}% Pneumonia)')
print('=' * 70)

In [ ]:
# --- CELL 3: Lung Bounding Boxes ---
from tqdm import tqdm

def compute_lung_bbox(image_path, margin=10):
    try:
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None: return None
        h, w = img.shape
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img_eq = clahe.apply(img)
        blurred = cv2.GaussianBlur(img_eq, (5, 5), 0)
        _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours: return None
        min_area = h * w * 0.05
        lung_contours = [c for c in contours if cv2.contourArea(c) > min_area]
        if not lung_contours:
            cx, cy = w // 2, h // 2
            crop_w, crop_h = int(w * 0.8), int(h * 0.8)
            return (cx - crop_w//2, cy - crop_h//2, cx + crop_w//2, cy + crop_h//2)
        all_points = np.vstack(lung_contours)
        x, y, bw, bh = cv2.boundingRect(all_points)
        return (max(0, x-margin), max(0, y-margin), min(w, x+bw+margin), min(h, y+bh+margin))
    except: return None

BBOX_SAVE_PATH = os.path.join(SAVE_DIR, 'lung_bboxes.csv')
if os.path.exists(BBOX_CSV):
    bbox_df = pd.read_csv(BBOX_CSV)
    print(f'Reusing precomputed bboxes: {BBOX_CSV} ({len(bbox_df)} records)')
elif os.path.exists(BBOX_SAVE_PATH):
    bbox_df = pd.read_csv(BBOX_SAVE_PATH)
    print(f'Loading existing bboxes: {BBOX_SAVE_PATH}')
else:
    print('Computing lung bounding boxes...')
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        bbox = compute_lung_bbox(row['image_path'])
        if bbox:
            records.append({'image_path': row['image_path'], 'x_min': bbox[0], 'y_min': bbox[1], 'x_max': bbox[2], 'y_max': bbox[3]})
        else:
            records.append({'image_path': row['image_path'], 'x_min': 0, 'y_min': 0, 'x_max': -1, 'y_max': -1})
    bbox_df = pd.DataFrame(records)
    bbox_df.to_csv(BBOX_SAVE_PATH, index=False)

bbox_lookup = bbox_df.set_index('image_path').to_dict('index')
print(f'Bounding boxes loaded in lookup: {len(bbox_lookup):,}')

In [5]:
# --- CELL 4: Compute Normalization Stats ---
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
means, stds = [], []
for idx in range(min(len(df), 200)):
    try:
        img = cv2.imread(df.iloc[idx]['image_path'], cv2.IMREAD_GRAYSCALE)
        img = clahe.apply(img)
        bb = bbox_lookup.get(df.iloc[idx]['image_path'], None)
        if bb and bb['x_max'] > 0:
            img = img[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
        img_arr = np.array(img, dtype=np.float32) / 255.0
        means.append(img_arr.mean())
        stds.append(img_arr.std())
    except: continue
MEAN = [np.mean(means)]
STD  = [np.mean(stds)]
print(f'MEAN: {MEAN[0]:.4f} | STD: {STD[0]:.4f}')

MEAN: 0.5007 | STD: 0.2732


In [6]:
# --- CELL 5: Transforms (Train / Val / TTA) ---

train_tfm = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.08)),
])

val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# TTA transforms — diverse but medically safe augmentations
tta_transforms = [
    val_tfm,  # 0: Original (no augmentation)
    transforms.Compose([  # 1: Slight rotation +5
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation((5, 5)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
    transforms.Compose([  # 2: Slight rotation -5
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation((-5, -5)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
    transforms.Compose([  # 3: Zoom in
        transforms.Resize((256, 256)),
        transforms.CenterCrop(210),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
    transforms.Compose([  # 4: Slight brightness up
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(brightness=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
    transforms.Compose([  # 5: Slight contrast
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
    transforms.Compose([  # 6: Slight translate
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomAffine(degrees=0, translate=(0.03, 0.03)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
    transforms.Compose([  # 7: Rotation +3
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation((3, 3)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
    transforms.Compose([  # 8: Rotation -3
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation((-3, -3)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
    transforms.Compose([  # 9: Gaussian blur
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.3, 0.5)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
]

print(f'Train transform: safe augmentations (no hflip)')
print(f'TTA transforms : {len(tta_transforms)} views')

Train transform: safe augmentations (no hflip)
TTA transforms : 10 views


In [ ]:
# --- CELL 6: Dataset Class ---
class EnhancedCXRDataset(Dataset):
    def __init__(self, df, bbox_lookup, transform=None):
        self.df = df.reset_index(drop=True)
        self.bbox_lookup = bbox_lookup or {}
        self.transform = transform
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        img = self.clahe.apply(img)
        bb = self.bbox_lookup.get(row.image_path, None)
        if bb and bb.get('x_max', 0) > 0:
            img = img[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
        img = Image.fromarray(img)
        if self.transform:
            img = self.transform(img)
        return img, int(row.label)

    def set_transform(self, transform):
        """Allows swapping transform for TTA."""
        self.transform = transform

print('EnhancedCXRDataset defined (with robust image loading & set_transform for TTA).')

In [8]:
# --- CELL 7: CBAM + Focal Loss + Mixup (same as v3) ---

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = x.mean(dim=1, keepdim=True)
        max_out = x.max(dim=1, keepdim=True)[0]
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention()
    def forward(self, x):
        return self.spatial_att(self.channel_att(x))

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.label_smoothing = label_smoothing
    def forward(self, logits, targets):
        num_classes = logits.shape[1]
        if self.label_smoothing > 0:
            smooth = torch.zeros_like(logits).fill_(self.label_smoothing / (num_classes - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
        else:
            smooth = F.one_hot(targets, num_classes).float()
        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)
        focal_weight = (1.0 - probs) ** self.gamma
        loss = -focal_weight * smooth * log_probs
        if self.alpha is not None:
            loss = loss * self.alpha.to(logits.device).unsqueeze(0)
        return loss.sum(dim=1).mean()

def mixup_data(x, y, alpha=0.3):
    lam = max(np.random.beta(alpha, alpha), 1 - np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[index], y, y[index], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print('CBAM, FocalLoss, Mixup defined.')

CBAM, FocalLoss, Mixup defined.


In [9]:
# --- CELL 8: Model Factory ---
def build_model():
    """Create a fresh model for each fold."""
    class EnhancedPneumoNetV4(nn.Module):
        def __init__(self):
            super().__init__()
            xrv_model = xrv.models.DenseNet(weights='densenet121-res224-all')
            self.features = xrv_model.features
            freeze_layers = {'conv0','norm0','relu0','pool0','denseblock1','transition1','denseblock2','transition2'}
            for name, child in self.features.named_children():
                for p in child.parameters():
                    p.requires_grad = name not in freeze_layers
            self.cbam = CBAM(1024, reduction=16)
            self.pool = nn.AdaptiveAvgPool2d(1)
            self.classifier = nn.Sequential(
                nn.Dropout(0.5), nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
                nn.Dropout(0.3), nn.Linear(512, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
                nn.Dropout(0.2), nn.Linear(128, NUM_CLASSES)
            )
        def forward(self, x):
            f = F.relu(self.features(x), inplace=True)
            f = self.cbam(f)
            f = self.pool(f).flatten(1)
            return self.classifier(f)
        def get_features(self, x):
            f = F.relu(self.features(x), inplace=True)
            f = self.cbam(f)
            return self.pool(f).flatten(1)

    model = EnhancedPneumoNetV4().to(DEVICE)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Params: {total:,} total, {trainable:,} trainable ({trainable/total*100:.1f}%)')
    return model

# Smoke test
print('Building model...')
_m = build_model()
_d = torch.randn(2, 1, 224, 224).to(DEVICE)
print(f'  Output: {_m(_d).shape}, Features: {_m.get_features(_d).shape}')
del _m, _d
torch.cuda.empty_cache()

Building model...
  Params: 7,670,756 total, 6,247,396 trainable (81.4%)
  Output: torch.Size([2, 2]), Features: torch.Size([2, 1024])


In [10]:
# --- CELL 9: Training + Evaluation Functions ---

def train_one_epoch(model, loader, criterion, optimizer, device, mixup_alpha):
    model.train()
    total_loss = total_correct = total_n = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        mixed, ya, yb, lam = mixup_data(imgs, labels, mixup_alpha)
        optimizer.zero_grad()
        out = model(mixed)
        loss = mixup_criterion(criterion, out, ya, yb, lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        preds = out.argmax(1)
        total_correct += (lam * (preds == ya).float().sum().item() +
                          (1 - lam) * (preds == yb).float().sum().item())
        total_n += len(labels)
    return total_loss / total_n, total_correct / total_n

def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            out = model(imgs)
            probs = F.softmax(out, dim=1)[:, 1]
            all_preds.extend(out.argmax(1).cpu().tolist())
            all_labels.extend(labels.tolist())
            all_probs.extend(probs.cpu().tolist())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    try: auc_val = roc_auc_score(all_labels, all_probs)
    except: auc_val = 0.0
    return acc, auc_val, f1, all_preds, all_labels, all_probs

def evaluate_tta(model, dataset, tta_tfms, device, batch_size=8):
    """Test Time Augmentation: average probabilities over multiple augmented views."""
    model.eval()
    n_samples = len(dataset)
    n_views = len(tta_tfms)
    all_probs = np.zeros((n_samples, n_views))
    all_labels = []

    for v, tfm in enumerate(tta_tfms):
        dataset.set_transform(tfm)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
        probs_v = []
        labels_v = []
        with torch.no_grad():
            for imgs, labels in loader:
                out = model(imgs.to(device))
                p = F.softmax(out, dim=1)[:, 1].cpu().numpy()
                probs_v.extend(p)
                labels_v.extend(labels.tolist())
        all_probs[:, v] = probs_v
        if v == 0:
            all_labels = labels_v

    # Average across TTA views
    avg_probs = all_probs.mean(axis=1)
    avg_preds = (avg_probs >= 0.5).astype(int)

    acc = accuracy_score(all_labels, avg_preds)
    f1  = f1_score(all_labels, avg_preds, average='macro')
    try: auc_val = roc_auc_score(all_labels, avg_probs)
    except: auc_val = 0.0
    return acc, auc_val, f1, avg_preds.tolist(), all_labels, avg_probs.tolist()

def find_optimal_threshold(labels, probs):
    fpr, tpr, thresholds = roc_curve(labels, probs)
    youden_j = tpr - fpr
    idx = np.argmax(youden_j)
    return thresholds[idx]

print('Training, Evaluation, TTA, Threshold functions defined.')

Training, Evaluation, TTA, Threshold functions defined.


In [ ]:
# --- CELL: Single Split Training (Experiment 1 — shared patient split) ---
# Cross-validation removed. All phases use identical train_df / val_df / test_df.

print('=' * 70)
print('SINGLE-SPLIT TRAINING (Patient-Level 70/15/15)')
print('=' * 70)
print(f'  Train: {len(train_df):,} rows | {train_df["subject_id"].nunique():,} patients')
print(f'  Val  : {len(val_df):,} rows | {val_df["subject_id"].nunique():,} patients')
print(f'  Test : {len(test_df):,} rows | {test_df["subject_id"].nunique():,} patients')

# Dataloaders
tr_labels  = train_df['label'].values
cls_counts = np.bincount(tr_labels)
sw         = [1.0 / cls_counts[l] for l in tr_labels]
sampler    = WeightedRandomSampler(sw, len(sw), replacement=True)

train_loader = DataLoader(EnhancedCXRDataset(train_df, bbox_lookup, train_tfm),
                          batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, drop_last=True)
val_loader   = DataLoader(EnhancedCXRDataset(val_df,   bbox_lookup, val_tfm),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_dataset = EnhancedCXRDataset(test_df, bbox_lookup, val_tfm)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Build model
model         = build_model()
class_weights = torch.tensor([1.0, PNEUMONIA_WT], dtype=torch.float32)
criterion     = FocalLoss(alpha=class_weights, gamma=2.0, label_smoothing=LABEL_SMOOTH)
eval_criterion = nn.CrossEntropyLoss()

pretrained_params = [p for n, p in model.named_parameters()
                     if p.requires_grad and 'classifier' not in n and 'cbam' not in n]
new_params        = [p for n, p in model.named_parameters()
                     if p.requires_grad and ('classifier' in n or 'cbam' in n)]
optimizer = optim.AdamW([
    {'params': pretrained_params, 'lr': LR},
    {'params': new_params,        'lr': LR * 10},
], weight_decay=WEIGHT_DECAY)

warmup    = LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
cosine    = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS, eta_min=1e-7)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

best_val_auc  = 0.0
best_state    = None
patience_cnt  = 0
history       = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': []}

for epoch in range(1, NUM_EPOCHS + 1):
    # ---- Train with Mixup ----
    model.train()
    ep_loss, ep_probs, ep_labels_ep = 0.0, [], []
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        mixed, ya, yb, lam = mixup_data(imgs, labels, MIXUP_ALPHA)
        optimizer.zero_grad()
        logits = model(mixed)
        loss   = mixup_criterion(criterion, logits, ya, yb, lam)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        
        probs = torch.softmax(logits.detach(), dim=1)[:, 1].cpu().numpy()
        ep_loss += loss.item() * labels.size(0)
        ep_probs.extend(probs)
        ep_labels_ep.extend(labels.cpu().tolist())
    scheduler.step()

    train_auc = roc_auc_score(ep_labels_ep, ep_probs) if len(set(ep_labels_ep)) > 1 else 0.5

    # ---- Validate ----
    model.eval()
    val_loss, val_probs, val_labels_ep = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = eval_criterion(logits, labels)
            probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            val_loss += loss.item() * labels.size(0)
            val_probs.extend(probs)
            val_labels_ep.extend(labels.cpu().tolist())

    val_auc = roc_auc_score(val_labels_ep, val_probs) if len(set(val_labels_ep)) > 1 else 0.5
    history['train_loss'].append(ep_loss / len(tr_labels))
    history['val_loss'].append(val_loss / len(val_df))
    history['train_auc'].append(train_auc)
    history['val_auc'].append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state   = copy.deepcopy(model.state_dict())
        patience_cnt = 0
    else:
        patience_cnt += 1

    print(f'  Ep {epoch:02d}/{NUM_EPOCHS:02d} | '
          f'Train Loss: {ep_loss/len(tr_labels):.4f}  AUC: {train_auc:.4f} | '
          f'Val Loss: {val_loss/len(val_df):.4f}  AUC: {val_auc:.4f} | '
          f'Best: {best_val_auc:.4f} (patience={patience_cnt}/{PATIENCE})')

    if patience_cnt >= PATIENCE:
        print(f'  Early stopping triggered at epoch {epoch}')
        break

print(f'\nTraining complete. Best Validation AUC: {best_val_auc:.4f}')
model.load_state_dict(best_state)

In [ ]:
# --- CELL: Test Evaluation & Results ---
# Evaluate best model on the held-out test set

model.load_state_dict(best_state)
model.eval()

test_probs, test_labels_list = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        test_probs.extend(probs)
        test_labels_list.extend(labels.cpu().tolist())

test_auc  = roc_auc_score(test_labels_list, test_probs)
test_preds = [1 if p >= 0.5 else 0 for p in test_probs]
test_acc   = accuracy_score(test_labels_list, test_preds)
test_f1    = f1_score(test_labels_list, test_preds)

# Youden-J threshold
fpr_v, tpr_v, thresh_v = roc_curve(test_labels_list, test_probs)
best_thresh = float(thresh_v[np.argmax(tpr_v - fpr_v)])
tuned_preds = [1 if p >= best_thresh else 0 for p in test_probs]
tuned_acc   = accuracy_score(test_labels_list, tuned_preds)
tuned_sens  = float(np.array(tuned_preds)[np.array(test_labels_list)==1].mean())
tuned_spec  = float((np.array(tuned_preds)==0)[np.array(test_labels_list)==0].mean())

print('=' * 60)
print('PHASE 1 — TEST SET RESULTS (Shared Split)')
print('=' * 60)
print(f'  Test AUC        : {test_auc:.4f}')
print(f'  Test Accuracy   : {test_acc:.4f}  (threshold=0.5)')
print(f'  Test F1         : {test_f1:.4f}')
print(f'  Youden Threshold: {best_thresh:.4f}')
print(f'  Tuned Accuracy  : {tuned_acc:.4f}')
print(f'  Sensitivity     : {tuned_sens:.4f}')
print(f'  Specificity     : {tuned_spec:.4f}')
print('=' * 60)
print(classification_report(test_labels_list, tuned_preds, target_names=['Normal','Pneumonia']))


In [ ]:
# --- CELL: ROC Curve & Confusion Matrix ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phase 1 — Image-Only Classifier (Shared Patient Split)', fontsize=13, fontweight='bold')

# ROC Curve
fpr_t, tpr_t, _ = roc_curve(test_labels_list, test_probs)
ax = axes[0]
ax.plot(fpr_t, tpr_t, lw=2, color='darkorange', label=f'AUC = {test_auc:.4f}')
ax.plot([0,1],[0,1],'navy',ls='--',lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Test Set'); ax.legend(); ax.grid(alpha=0.3)

# Confusion Matrix
cm = confusion_matrix(test_labels_list, tuned_preds)
ax = axes[1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal','Pneumonia'], yticklabels=['Normal','Pneumonia'])
ax.set_title(f'Confusion Matrix (thresh={best_thresh:.3f})')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'phase1_roc_cm.png'), dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# --- CELL: TTA Evaluation ---
# Average predictions over 10 augmented views of each test image

from src.data.preprocessing import get_tta_transforms

tta_tfms = get_tta_transforms(img_size=IMG_SIZE)
tta_all_probs = []

model.eval()
for tfm in tta_tfms:
    test_dataset.set_transform(tfm)
    view_probs = []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            view_probs.extend(probs)
    tta_all_probs.append(view_probs)

# Restore original transform
test_dataset.set_transform(val_tfm)

tta_probs  = np.mean(tta_all_probs, axis=0)
tta_auc    = roc_auc_score(test_labels_list, tta_probs)
tta_preds  = [1 if p >= 0.5 else 0 for p in tta_probs]
tta_acc    = accuracy_score(test_labels_list, tta_preds)

print('=' * 60)
print('TTA RESULTS (10-view average)')
print('=' * 60)
print(f'  TTA AUC      : {tta_auc:.4f}  (vs standard {test_auc:.4f})')
print(f'  TTA Accuracy : {tta_acc:.4f}')
print('=' * 60)

# TTA ROC
fpr_tta, tpr_tta, _ = roc_curve(test_labels_list, tta_probs)
plt.figure(figsize=(6,5))
plt.plot(fpr_tta, tpr_tta, lw=2, color='green', label=f'TTA AUC = {tta_auc:.4f}')
plt.plot(fpr_t, tpr_t, lw=2, color='darkorange', ls='--', label=f'Standard AUC = {test_auc:.4f}')
plt.plot([0,1],[0,1],'navy',ls='--',lw=1)
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC — Standard vs TTA')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(SAVE_DIR, 'phase1_tta_roc.png'), dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# --- CELL: Save Results ---
import json as json_lib

final_results = {
    'experiment': 'Experiment 1 — Shared Patient-Level Split',
    'phase': '1 — Image Only',
    'split': {
        'n_train': len(train_df), 'n_val': len(val_df), 'n_test': len(test_df),
        'train_patients': int(train_df['subject_id'].nunique()),
        'val_patients':   int(val_df['subject_id'].nunique()),
        'test_patients':  int(test_df['subject_id'].nunique()),
    },
    'hyperparams': {
        'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE,
        'num_epochs': NUM_EPOCHS, 'lr': LR,
        'tta_views': TTA_TRANSFORMS, 'seed': SEED,
    },
    'results': {
        'test_auc':   round(test_auc, 4),
        'test_acc':   round(test_acc, 4),
        'test_f1':    round(test_f1, 4),
        'tta_auc':    round(float(tta_auc), 4),
        'tta_acc':    round(float(tta_acc), 4),
        'youden_threshold': round(best_thresh, 4),
        'tuned_acc':  round(tuned_acc, 4),
        'sensitivity': round(tuned_sens, 4),
        'specificity': round(tuned_spec, 4),
    },
}

out_path = os.path.join(SAVE_DIR, 'phase1_results.json')
with open(out_path, 'w') as f:
    json_lib.dump(final_results, f, indent=2)

# Save model checkpoint
ckpt_path = os.path.join(SAVE_DIR, 'phase1_best_model.pth')
torch.save(best_state, ckpt_path)

print('=' * 60)
print('Phase 1 Complete')
print('=' * 60)
print(f'  Test AUC      : {test_auc:.4f}')
print(f'  TTA AUC       : {tta_auc:.4f}')
print(f'  Sensitivity   : {tuned_sens:.4f}')
print(f'  Specificity   : {tuned_spec:.4f}')
print(f'  Results saved : {out_path}')
print(f'  Model saved   : {ckpt_path}')


## 📊 Experiment 1 — Phase 1 Performance Summary (Canonical Patient-Level Split)

### 1. Cohort & Patient-Level Split (Zero Leakage)
*   **Split Methodology**: Stratified patient-level partition using `subject_id` (70% train / 15% val / 15% test).
*   **Train Set**: 2,635 images across **2,382 patients** (1,311 Normal, 1,324 Pneumonia — 50.2% positive).
*   **Val Set**: 566 images across **511 patients** (278 Normal, 288 Pneumonia — 50.9% positive).
*   **Test Set**: 562 images across **511 patients** (283 Normal, 279 Pneumonia — 49.6% positive).
*   **Patient Overlap**: **0 patients** appear across multiple splits (100% clean hold-out).

---

### 2. Quantitative Results on Test Cohort (562 CXR Images)

| Metric | Default ($	au = 0.500$) | Youden-J Tuned ($	au = 0.535$) | 10-View TTA Averaged |
| :--- | :---: | :---: | :---: |
| **ROC AUC** | **0.8408** | **0.8408** | **0.8446** (+0.0038) |
| **Accuracy** | 70.46% | **76.51%** (+6.05%) | 69.22% |
| **Sensitivity (Recall)** | 85.30% | **77.06%** | 83.15% |
| **Specificity** | 55.83% | **75.97%** (+20.14%) | 55.48% |
| **Macro F1 Score** | 0.7492 | **0.7648** | 0.7381 |

---

### 3. Key Observations & Findings
1.  **Elimination of Overoptimistic Bias**: Previous unstratified or row-level splits risked reporting artificially inflated metrics due to identical patients leaking across folds. On this strict patient-level test set, the image-only backbone achieves a genuine **0.8408 Standard AUC** and **0.8446 TTA AUC**.
2.  **Youden's J Optimization**: Adjusting the decision threshold from 0.500 to **0.5347** rebalances the classifier, lifting specificity from a poor 55.83% to **75.97%** while maintaining a clinically viable **77.06% sensitivity**, boosting overall accuracy to **76.51%**.
3.  **TTA Robustness**: Averaging over 10 medically safe geometric/photometric augmentations delivers a solid **+0.38% AUC gain** without requiring additional model parameters.
